# Student Performance — Data Generation (Multivariate)

this notebook loads the Student Performance dataset, encodes categorical columns, and generates three datasets with MCAR, MAR, and MNAR missingness using mdatagen.

run this notebook once to reproduce the CSV files used by `../analysis_multi.ipynb`. the CSVs are saved in the same `data/` folder as this notebook.

In [8]:
import pandas as pd
import numpy as np
from ucimlrepo import fetch_ucirepo

from mdatagen.multivariate.mMCAR import mMCAR
from mdatagen.multivariate.mMAR  import mMAR
from mdatagen.multivariate.mMNAR import mMNAR

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [9]:
# initial data set loading
student_performance = fetch_ucirepo(id=320)
X_raw = student_performance.data.features
y = student_performance.data.targets.iloc[:, 0].to_numpy()

X_raw.isna().sum()

X_raw.shape

school        0
sex           0
age           0
address       0
famsize       0
Pstatus       0
Medu          0
Fedu          0
Mjob          0
Fjob          0
reason        0
guardian      0
traveltime    0
studytime     0
failures      0
schoolsup     0
famsup        0
paid          0
activities    0
nursery       0
higher        0
internet      0
romantic      0
famrel        0
freetime      0
goout         0
Dalc          0
Walc          0
health        0
absences      0
dtype: int64

(649, 30)

The dataset is already complete with no missing values, so we can use it directly.

We encode all categorical columns as integer codes so that every column is numeric, required by the mdatagen library.

We use `G1` (the first-period grade) as the target variable `y`. It is passed to the generators because the library requires it — `G1` is not a feature in `X` and will not receive any missing values.

In [10]:
X = X_raw.copy()
cat_cols = X.select_dtypes(include=["object", "category"]).columns
for col in cat_cols:
    X[col] = pd.factorize(X[col])[0]
X = X.astype(float)

X.isna().sum()

X.shape

school        0
sex           0
age           0
address       0
famsize       0
Pstatus       0
Medu          0
Fedu          0
Mjob          0
Fjob          0
reason        0
guardian      0
traveltime    0
studytime     0
failures      0
schoolsup     0
famsup        0
paid          0
activities    0
nursery       0
higher        0
internet      0
romantic      0
famrel        0
freetime      0
goout         0
Dalc          0
Walc          0
health        0
absences      0
dtype: int64

(649, 30)

Now we have a clean numeric dataset to work with. We inject missing values into multiple columns simultaneously to simulate realistic multivariate missingness patterns.

### MCAR generation

Here we'll generate a dataset with MCAR missingness across multiple columns simultaneously.

Missingness is completely random and does not depend on any observed variable or on `G1`.

In [11]:
np.random.seed(42)
df_mcar = mMCAR(X=X, y=y, missing_rate=30).random()

df_mcar.isna().sum()

df_mcar.shape

school        206
sex           198
age           198
address       197
famsize       201
Pstatus       199
Medu          192
Fedu          182
Mjob          192
Fjob          195
reason        189
guardian      175
traveltime    183
studytime     201
failures      210
schoolsup     186
famsup        207
paid          201
activities    186
nursery       201
higher        207
internet      184
romantic      184
famrel        184
freetime      198
goout         205
Dalc          191
Walc          184
health        196
absences      209
dtype: int64

(649, 30)

### MAR generation

Here we'll generate a dataset with MAR missingness across multiple columns simultaneously.

Missingness is introduced using the random method from mdatagen. In `n_xmiss=15` randomly selected column pairs, the rows with the lowest values of the observed column are made missing in the other column — so missingness depends on observed values, not on the missing value itself.

In [12]:
np.random.seed(42)
df_mar = mMAR(X=X, y=y, n_xmiss=15).random(missing_rate=30)

df_mar.isna().sum()

df_mar.shape

school          0
sex             0
age             0
address       389
famsize         0
Pstatus       389
Medu            0
Fedu          389
Mjob          389
Fjob          389
reason        389
guardian      389
traveltime    389
studytime     389
failures        0
schoolsup       0
famsup        389
paid            0
activities      0
nursery         0
higher          0
internet        0
romantic      389
famrel        389
freetime      389
goout         389
Dalc          389
Walc            0
health          0
absences        0
target          0
dtype: int64

(649, 31)

### MNAR generation

Here we'll generate a dataset with MNAR missingness across multiple columns simultaneously.

Missingness is generated on the prediction target `G1` using `threshold=1`. Students with the lowest academic performance are most likely to have their feature records missing, simulating the case where the worst-performing students are systematically underrepresented in the data.

Since `G1` is the target (`y`) and is absent from the generated feature matrix, its effect must be inferred through correlated proxies such as `failures`, `absences`, and `studytime`.

In [13]:
np.random.seed(42)
df_mnar = mMNAR(X=X, y=y, threshold=1, n_xmiss=15).random(missing_rate=30)

df_mnar.isna().sum()

df_mnar.shape

school        389
sex           389
age             0
address       389
famsize       389
Pstatus       389
Medu            0
Fedu            0
Mjob          389
Fjob            0
reason          0
guardian      389
traveltime      0
studytime       0
failures        0
schoolsup     389
famsup        389
paid          389
activities    389
nursery         0
higher          0
internet      389
romantic        0
famrel        389
freetime        0
goout         389
Dalc            0
Walc            0
health          0
absences      389
target          0
dtype: int64

(649, 31)

Now we save the generated datasets to CSV so the analysis notebook can load them any time.

In [14]:
df_mcar.to_csv("mcar_multi.csv", index=False)
df_mar.to_csv("mar_multi.csv", index=False)
df_mnar.to_csv("mnar_multi.csv", index=False)